# Prompt Engineering

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained("microsoft/Phi-3-mini-4k-instruct",

                                            device_map="cuda",
                                            torch_dtype="auto",
                                            trust_remote_code=False)

tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

# Create a pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=500,
    do_sample=False,
)

d:\2026-courses\LLMs-Handson\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 195/195 [00:13<00:00, 14.73it/s]
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [2]:
# Prompt
messages = [
    {
        "role": "user",
        "content": "create a funny joke about chikens"
    }
]
output = pipe(messages)
print(output)


Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': ' Why did the chicken join the band? Because it had the drumsticks!'}]


transfomers.pipeline first converts our messages into a specific prompt template. We can explore this process be accessing the underlying tokenizer:


In [5]:
prompt = pipe.tokenizer.apply_chat_template(
    messages,
    tokenize=False
)

print(prompt)

<|user|>
create a funny joke about chikens<|end|>
<|endoftext|>


In [16]:
# Using temperature
output= pipe(messages, do_sample=True, temperature=1)
print(output[0]["generated_text"])

Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 Why don't chickens wear socks? Because they don't want to spoil their own eggs!


- Temperature parameter controls the randomness or creativity of the text generated. It defines how likely it is to choose tokens that are less probable.
- The underlying idea is that temperature of 0 generates the same response every time because it always chooses the most liekly word.
- a higher temperature (e.g., 0.8) generally results in a more diverse output while a lower temperature (e.g., 0.2) creates a more deterministic output.

## top_p
- top_p, also known as nucleus sampling, is sampling technique that controls which subset of tokens  (the nucleus) the LLM can consider.

In [18]:
output= pipe(messages, do_sample=True, top_p=1)
print(output[0]["generated_text"])

Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 Why did the chicken join a band? Because it wanted to make everyone's day with its pecking-tolerance!


- Be Creative = High temperature and top_p
- Being predictable= lower temperature and top_p